In [2]:
import duckdb
import polars as pl
import numpy as np
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)

print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df = df.with_columns(
    artist_name = pl.col("artist_name").str.to_lowercase(),
    track_name = pl.col("track_name").str.to_lowercase(),
    album_name = pl.col("album_name").str.to_lowercase()
)
df.head()

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
"""aldn""","""fa01a34e-05f3-424c-8fd0-390840…","""ifls""","""15020146-9301-457c-b45e-f24504…","""ifls""","""cb2297f2-a1fc-47f0-9f4a-1d0780…",1665215990,2022-10-08 07:59:50
"""dijon""","""cceea21d-f3ec-4586-be7a-d2e939…","""absolutely""","""7fc99d84-befc-4fca-b29f-a5711a…","""big mike's""","""1ca0c4d4-5dc8-4e4d-9109-2d3eff…",1709817266,2024-03-07 13:14:26
"""joji""","""8264722b-df00-467a-858e-5c97cd…","""smithereens""","""e26e8cbc-db4c-422a-a171-f0080a…","""night rider""","""""",1667672040,2022-11-05 18:14:00
"""tyler, the creator""","""""","""chromakopia""","""""","""like him (feat. lola young)""","""""",1738857239,2025-02-06 15:53:59
"""dimitri vegas & like mike""","""c5931849-7c9d-465e-b717-29e2af…","""disco:wax presents: remixes pa…","""""","""stampede - major lazer x p.a.f…","""""",1657791513,2022-07-14 09:38:33


## Seasonal patterns — do you listen to certain artists more in winter vs. summer?


In [14]:
df_ = df.filter(pl.col("track_played_utc").dt.year() == 2025)

In [24]:
df_pivot = (
    df_
    .group_by(
        pl.col("artist_name"), 
        pl.col("track_played_utc").dt.strftime("%B").alias("month"),
        pl.col("track_played_utc").dt.month().alias("month_idx")
    )
    .agg(
        scrobbles = pl.len()
    )
    .sort(pl.col("month_idx"), descending=False)
    .pivot(
        on="month",
        index="artist_name",
        values="scrobbles",
        aggregate_function="sum"
    )
)
df_pivot

artist_name,January,February,March,April,May,June,July,August,September,October,November,December
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""oklou""",20,43,5,3,3,1,1,1,0,3,6,1
"""frost children""",6,2,5,4,4,1,5,1,1,0,0,0
"""tchami""",1,0,0,0,0,0,0,0,0,0,0,0
"""mad keys""",20,6,2,2,0,1,0,0,0,0,0,0
"""pretty sick""",2,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…
"""skee mask""",0,0,0,0,0,0,0,0,0,0,0,1
"""vjac0b""",0,0,0,0,0,0,0,0,0,0,0,1
"""black tape for a blue girl""",0,0,0,0,0,0,0,0,0,0,0,1


In [29]:
n = 20
df_top_n_artists = (
    df_
    .group_by(
        pl.col("artist_name"), 
        # pl.col("track_played_utc").dt.strftime("%B").alias("month"),
        # pl.col("track_played_utc").dt.month().alias("month_idx")
    )
    .agg(
        scrobbles = pl.len()
    )
    .sort(pl.col("scrobbles"), descending=True)
).head(20)

df_pivot.join(df_top_n_artists, on="artist_name").sort("scrobbles", descending=True)

artist_name,January,February,March,April,May,June,July,August,September,October,November,December,scrobbles
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""bladee""",26,53,31,23,68,25,38,79,50,42,11,13,459
"""dean blunt""",0,7,86,10,27,46,7,82,58,66,8,14,411
"""bassvictim""",5,90,35,35,12,5,20,0,3,65,73,26,369
"""astrid sonne""",4,2,36,11,159,28,8,11,21,9,8,4,301
"""vegyn""",5,6,2,153,23,68,4,12,3,10,11,4,301
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mj lenderman""",0,6,129,9,5,6,0,5,4,5,1,4,174
"""jane remover""",36,22,17,73,9,0,3,1,1,0,3,2,167
"""blood orange""",0,0,0,0,0,27,41,50,31,4,3,10,166


## "Throwback" plays — replaying old favorites after a long gap

In [ ]:
# # Cumulative listening per track, so each record gets a number representing the n'th time of listening to a track.
# bv = (
#     df.sort(pl.col("date_played_unix"), descending=False)
#     .filter(pl.col("track_played_utc").dt.year() == year)
#     .filter(pl.col("artist_name") == "bassvictim")
#     .filter(pl.col("track_name").len().over("track_name") > 10)
#     .with_columns(
#         track_nth_scrobble = pl.col("track_name").cum_count().over(pl.col("track_name"))
#     )
# )
# bv.head()

In [47]:
(
    df
    .sort(pl.col("track_played_utc"), descending=False)
    .with_columns(
        last_played_delta = (
            pl.col("track_played_utc") - pl.col("track_played_utc").shift(1).over("track_name")
            ).dt.total_days(fractional=True).fill_null(0).ceil()
    )
).filter(pl.col("artist_name") == "bladee")

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,last_played_delta
str,str,str,str,str,str,i64,datetime[μs],f64
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""drama (feat. charli xcx)""","""""","""drama (feat. charli xcx)""","""""",1621494114,2021-05-20 07:01:54,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""gluee""","""250bafde-fd0c-4490-a728-5f741f…","""ebay""","""""",1621494495,2021-05-20 07:08:15,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""working on dying""","""587b563c-73ff-4d81-b669-b49a2e…","""lordship""","""22de2b38-a027-49fe-8e99-c30809…",1621494644,2021-05-20 07:10:44,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""red light""","""725f97cb-5e7a-4bbb-be43-c65004…","""decay""","""404d136c-174c-4cb5-9fb0-1e8859…",1621494772,2021-05-20 07:12:52,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""working on dying""","""587b563c-73ff-4d81-b669-b49a2e…","""lordship""","""22de2b38-a027-49fe-8e99-c30809…",1621498932,2021-05-20 08:22:12,1.0
…,…,…,…,…,…,…,…,…
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""love is a state""","""3540f010-4c99-4c9e-80e1-3dba8f…","""eyelash""","""6253d641-5088-4908-96df-100b4e…",1774283687,2026-03-23 16:34:47,0.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""eversince""","""96e5ccf7-2ba9-4f94-944d-79ca01…","""rip""","""ecd0d043-fa8e-44e1-926b-94e108…",1774283914,2026-03-23 16:38:34,517.0
"""bladee""","""cd689e77-dfdd-4f81-b50c-5e5a3f…","""sesame street""","""2c0a269b-535a-47fe-97b2-fa4d65…","""sesame street""","""1907b9ad-2964-4889-9df3-be44ae…",1774355995,2026-03-24 12:39:55,26.0
